In [48]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import sqlite3
import pandas as pd
import os

DB_PATH = db_path = r"H:\ua-aid-intelligence-hub\data\master\master.db"

def get_donations_in_eur():
    """
    Extracts all donations, converts UAH to EUR, retains the exchange rate,
    and drops the original UAH amount. Preserves all other metadata.
    """
    query = """
    SELECT
        d.id,
        d.date,
        d.foundation_name,
        d.category,
        ROUND(d.amount / er.rate_uah, 2) AS amount_eur,
        er.rate_uah AS eur_exchange_rate,
        d.currency AS original_currency,
        d.comment,
        d.source AS donation_source
    FROM donations d
    LEFT JOIN exchange_rates er
        ON d.date = er.date AND er.currency = 'EUR'
    ORDER BY d.date DESC;
    """

    try:
        with sqlite3.connect(DB_PATH) as conn:
            df_donations = pd.read_sql_query(query, conn)
            # Convert date strings to datetime objects for time-series analysis
            df_donations['date'] = pd.to_datetime(df_donations['date'])
            return df_donations
    except Exception as e:
        print(f"Database extraction failed: {e}")
        return pd.DataFrame()

def get_daily_aggregated_news():
    """
    Extracts news data and aggregates multiple headlines and sources
    per day into single strings to prevent row duplication upon merging.
    """
    query = """
    SELECT
        date,
        GROUP_CONCAT(source, ' | ') AS daily_sources,
        GROUP_CONCAT(headers, ' | ') AS daily_headers
    FROM news
    GROUP BY date
    ORDER BY date DESC;
    """

    try:
        with sqlite3.connect(DB_PATH) as conn:
            df_news = pd.read_sql_query(query, conn)
            # Convert date strings to datetime objects to ensure clean merging later
            df_news['date'] = pd.to_datetime(df_news['date'])
            return df_news
    except Exception as e:
        print(f"Database extraction failed: {e}")
        return pd.DataFrame()

# 1. Load donations dataset
df_donations = get_donations_in_eur()
print(f"Donations dataset loaded: {len(df_donations)} rows.")

# 2. Load aggregated news dataset
df_news = get_daily_aggregated_news()
print(f"News dataset loaded: {len(df_news)} rows.")

# Display settings for clear data visualization in Jupyter
pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.max_colwidth = 80

In [ ]:
df_donations.head()

In [ ]:
df_news.head()

In [ ]:
df = df_donations.copy()

The goal is to study "grassroots" donations by filtering for transactions under €10,000.

Since we have transaction-level data only for the 'Come Back Alive' (CBA) foundation, while United24 provides only daily aggregates (which would be automatically filtered out by this limit), I am focusing exclusively on CBA. By looking only at payments under €10k, I want to analyze the behavior of "ordinary" people rather than large organizations or funds.

In [ ]:
df = df.query('foundation_name == "come_back_alive" and amount_eur <= 10000')

In [ ]:
df.dtypes

In [ ]:
dayly = df.groupby('date', as_index = False)\
        .agg({'amount_eur':'sum'})
dayly.head()

In [ ]:
sns.lineplot(dayly, y = 'amount_eur', x = 'date')

Anomaly Detection: Negative Values
A quick look at the plot reveals a sharp drop into negative numbers. This is highly unusual behavior for donations. Let's dig deeper to investigate these negative records.

In [ ]:
dayly[dayly.amount_eur < 0]

In [ ]:
df[df.date == '2025-04-08'].amount_eur.hist()

In [ ]:
df[(df['date'] == '2025-04-08') & (df['amount_eur'] < 0)]

Fortunately, there are only two records with negative donation amounts. This makes it easy to look into the raw data and find out exactly what caused the graph to anomalously drop below zero.

In [ ]:
df.loc[1524092, 'comment']

Translation
"Transfer of funds according to Addendum No. 2 dated March 27, 2025, VAT-exempt - PJSC UKRNAFTA"

Conclusion: This is a corporate accounting correction (likely a refund) from a major oil company. It is a technical transaction, not a regular grassroots donation, and must be removed from our dataset.

In [ ]:
df = df[df['amount_eur'] >= 0]

In [ ]:
#lets's look at the gpaph now after I fixed it
dayly = df.groupby('date', as_index = False)\
        .agg({'amount_eur':'sum'})

sns.lineplot(dayly, y = 'amount_eur', x = 'date')

Okay, now everything is fine, we can start exploring public donations.